# Notebook 1: Testing jev.decide

This notebook is the second layer of testing for `jev.decide`, the Neo4j function built in this book. The first layer is the unit tests that run with `mvn package`, which use a fake network. This one runs against your real database, through Cypher, the way you'd use the function.

Each test is a Cypher call. The notebook checks the result and prints PASS or FAIL. At the end it saves everything to `results_decide.json`.

This is the version for chapter 5. It covers registration, validation, a closed port, the rule for where the API key goes and, optionally, the hosted Jev API. Chapter 7 adds tests for a local model.

A few design choices are worth knowing about:

- **Nothing is written to the database.** Every test is a read-only `RETURN`.
- **Four statuses.** `PASS` and `FAIL` are checks with a clear right answer. `INFO` records something we measured, like a latency. `SKIPPED` means a test couldn't run.
- **The hosted API is off by default.** `RUN_REAL_JEV` is `False`, so a plain run makes no paid calls.
- **A re-run replaces earlier results.** Run any test cell again and its entry is updated, not duplicated.

## Prerequisites

Set the following environment variables before launching Jupyter. The URI is the local Neo4j Desktop one:

```shell
export NEO4J_URI="bolt://localhost:7687"
export NEO4J_USERNAME="neo4j"
export NEO4J_PASSWORD="your-password"
export NEO4J_DATABASE="neo4j"
```

You also need:

- The plugin JAR in the `plugins` folder of your DBMS, with `dbms.security.procedures.allowlist=jev.*` in `neo4j.conf`.
- For the hosted API only, `server.jvm.additional=-DTYPESAFE_API_KEY=<your-api-key>` in `neo4j.conf`. The notebook never sees the key.

## 1. Install Dependencies

In [1]:
%pip install neo4j==5.28.1 \
             pandas==2.2.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import json
import os
import pandas as pd
import statistics
import time

from datetime import datetime
from neo4j import GraphDatabase
from pathlib import Path

## 3. Configuration

`RUN_REAL_JEV` decides whether the optional hosted-API section runs. Leave it `False` unless the key is in `neo4j.conf` and you're happy to spend a few calls.

`STATE` and `QUESTIONS` are the fixture used by the tests: one transaction and two questions, a choice and a self-rating.

In [3]:
NEO4J_URI      = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]

print("Credentials set.")

Credentials set.


In [4]:
RUN_REAL_JEV  = False   # True: also test the hosted Jev API
REAL_JEV_RUNS = 5       # calls to the hosted API when RUN_REAL_JEV is True

CLOSED_URL  = "http://127.0.0.1:9/v1/systemone"   # nothing listens on port 9

RESULTS_FILE = "results_decide.json"

STATE = {
    "amount": 551.89,
    "merchant_category": "atm_withdrawal",
    "distance_from_home_km": 204.9,
    "hour_of_day": 0,
}

QUESTIONS = {
    "decision": {
        "type": "choice",
        "instructions": "Should this transaction be flagged for review or passed through?",
        "criteria": {
            "Flag": "Potential fraud. Send for review.",
            "Pass": "Likely legitimate. Process normally.",
        },
    },
    "self_rating": {
        "type": "score",
        "instructions": "How confident are you in this decision?",
        "criteria": ["Not confident", "Somewhat confident", "Very confident"],
    },
}

print(f"RUN_REAL_JEV = {RUN_REAL_JEV}")
print("Configuration set.")

RUN_REAL_JEV = False
Configuration set.


## 4. Connect to Neo4j

In [5]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Test Helpers

`check`, `observe` and `skip` record a result and print it. `run_decide` makes one `jev.decide` call and returns the result map together with the wall-clock time in milliseconds. That time includes the trip over Bolt and the Cypher planning, so it's always a little more than `metadata.latency_ms`, which the function measures itself.

In [6]:
RESULTS = []   # one entry per test
TIMINGS = {}   # raw numbers, saved next to the results

def record(name, status, detail=""):
    """Store a result. Running a test again replaces its earlier entry."""
    RESULTS[:] = [r for r in RESULTS if r["test"] != name]
    RESULTS.append({"test": name, "status": status, "detail": str(detail)})
    print(f"{status:8} {name}" + (f" -- {detail}" if detail != "" else ""))

def check(name, condition, detail=""):
    record(name, "PASS" if condition else "FAIL", detail)

def observe(name, detail):
    record(name, "INFO", detail)

def skip(name, reason):
    record(name, "SKIPPED", reason)

def starts(text, prefix):
    return isinstance(text, str) and text.startswith(prefix)

def decision_of(result, question="decision"):
    return ((result.get("answers") or {}).get(question)) or {}

DECIDE_QUERY = "RETURN jev.decide($state, $questions, $options) AS result"

def run_decide(state, questions=None, options=None):
    """One jev.decide call. Returns the result map and the wall-clock milliseconds."""
    params = dict(
        state=state,
        questions=QUESTIONS if questions is None else questions,
        options=options or {},
    )
    with driver.session(database=NEO4J_DATABASE) as session:
        started = time.perf_counter()
        result = session.run(DECIDE_QUERY, **params).single()["result"]
        elapsed = (time.perf_counter() - started) * 1000
    return result, elapsed

def save_results():
    payload = {
        "notebook":  "01_decide_tests",
        "saved_at":  datetime.now().isoformat(timespec="seconds"),
        "results":   RESULTS,
        "timings":   TIMINGS,
    }
    Path(RESULTS_FILE).write_text(json.dumps(payload, indent=2, default=str))
    print(f"Saved {len(RESULTS)} results to {RESULTS_FILE}")

print("Helpers defined.")

Helpers defined.


## 6. The Tests

These tests never leave the database, apart from the closed-port tests, which fail on purpose, and the optional hosted section at the end.

### The function is installed

If this fails, the JAR isn't in the `plugins` folder, the database wasn't restarted or the allowlist line is missing.

In [7]:
with driver.session(database=NEO4J_DATABASE) as session:
    names = [r["name"] for r in session.run(
        "SHOW FUNCTIONS YIELD name WHERE name STARTS WITH 'jev' RETURN name"
    )]

check("function jev.decide is registered", "jev.decide" in names, ", ".join(names))

PASS     function jev.decide is registered -- jev.decide


### Validation stops bad input before any call

The function checks its input first. Each case below should come back with an error starting `validation:`, an answer of `null` and zero attempts, which proves no request was made. The first case is the most common mistake: an array of criteria for a choice question, where a map is needed.

In [8]:
bad_questions = {"d": {"type": "choice", "instructions": "Pick one", "criteria": ["a", "b"]}}
result, ms = run_decide("some text", bad_questions)

check("validation: choice criteria as an array",
      starts(result["error_message"], "validation:"), result["error_message"])
check("validation: no call was made", result["metadata"]["attempts"] == 0)
check("validation: answers are null", result["answers"] is None)

PASS     validation: choice criteria as an array -- validation: d: a choice question needs criteria as a map of label to description, not an array
PASS     validation: no call was made
PASS     validation: answers are null


In [9]:
VALIDATION_CASES = [
    ("unknown option name",       "some text", None, {"modle": "x"}),
    ("blank state",               "   ",       None, {}),
    ("score criteria as a map",   "some text",
        {"s": {"type": "score", "instructions": "Rate it", "criteria": {"a": "b"}}}, {}),
    ("retries out of range",      "some text", None, {"max_retries": 99}),
    ("missing instructions",      "some text",
        {"d": {"type": "choice", "criteria": {"A": "a", "B": "b"}}}, {}),
    ("url that is not http",      "some text", None, {"url": "ftp://example.com/x"}),
    ("no questions at all",       "some text", {}, {}),
]

for label, state, questions, options in VALIDATION_CASES:
    result, ms = run_decide(state, questions, options)
    ok = starts(result["error_message"], "validation:") and result["metadata"]["attempts"] == 0
    check(f"validation: {label}", ok, result["error_message"])

PASS     validation: unknown option name -- validation: unknown option(s): modle; known options are [connect_timeout_ms, max_retries, model, request_timeout_ms, url]
PASS     validation: blank state -- validation: state must not be blank
PASS     validation: score criteria as a map -- validation: s: a score question needs criteria as an ordered array of level descriptions, not a map
PASS     validation: retries out of range -- validation: max_retries must be between 0 and 5
PASS     validation: missing instructions -- validation: d: instructions are required
PASS     validation: url that is not http -- validation: url must be an http or https URL with a host
PASS     validation: no questions at all -- validation: questions must be a non-empty map


### A closed port gives a clear error

Nothing listens on port 9, so the connection is refused. The error should name the host and port, so a wrong `url` is easy to spot. With `max_retries` of 0 there's one attempt. With the default of 2 there are three, with waits of 0.5 s and 1 s between them, so the call takes at least 1.5 s.

In [10]:
result, ms = run_decide(STATE, options={"url": CLOSED_URL, "max_retries": 0})
check("closed port: error names the host and port",
      starts(result["error_message"], "io: ConnectException reaching 127.0.0.1:9"),
      result["error_message"])
check("closed port: one attempt without retries", result["metadata"]["attempts"] == 1)
check("closed port: the key was not sent", result["metadata"]["authenticated"] is False)

result, ms = run_decide(STATE, options={"url": CLOSED_URL})
check("closed port: three attempts with default retries",
      result["metadata"]["attempts"] == 3 and "after 3 attempts" in str(result["error_message"]),
      result["error_message"])
check("closed port: backoff waits happened (0.5 s + 1 s)",
      result["metadata"]["latency_ms"] >= 1400, f"{result['metadata']['latency_ms']} ms")

PASS     closed port: error names the host and port -- io: ConnectException reaching 127.0.0.1:9 (after 1 attempt)
PASS     closed port: one attempt without retries
PASS     closed port: the key was not sent
PASS     closed port: three attempts with default retries -- io: ConnectException reaching 127.0.0.1:9 (after 3 attempts)
PASS     closed port: backoff waits happened (0.5 s + 1 s) -- 1516 ms


### The API key goes only to the real host

The key is sent only to `https://api.typesafe.ai`. These look-alike addresses must not receive it. They end in `.invalid`, a name reserved so it never resolves, so the tests send nothing to a real server. The first adds a suffix to the right name. The second hides the real host behind a user name, so the host Java connects to is `lookalike.invalid`.

`metadata.authenticated` should be `False` for both. The exact error text depends on your network, so we record it for information.

In [11]:
LOOKALIKES = [
    ("suffix on the real name", "https://api.typesafe.ai.invalid/v1/systemone"),
    ("real name as a user name", "https://api.typesafe.ai@lookalike.invalid/v1/systemone"),
]

for label, url in LOOKALIKES:
    result, ms = run_decide(STATE, options={"url": url, "max_retries": 0})
    check(f"key rule: not sent to a look-alike ({label})",
          result["metadata"]["authenticated"] is False)
    check(f"key rule: no key was needed ({label})",
          not starts(result["error_message"], "no_api_key:"))
    observe(f"key rule: error text ({label})", result["error_message"])

PASS     key rule: not sent to a look-alike (suffix on the real name)
PASS     key rule: no key was needed (suffix on the real name)
INFO     key rule: error text (suffix on the real name) -- io: ConnectException reaching api.typesafe.ai.invalid (after 1 attempt)
PASS     key rule: not sent to a look-alike (real name as a user name)
PASS     key rule: no key was needed (real name as a user name)
INFO     key rule: error text (real name as a user name) -- io: ConnectException reaching lookalike.invalid (after 1 attempt)


### Optional: the hosted Jev API

Runs only when `RUN_REAL_JEV` is `True`. It needs `TYPESAFE_API_KEY` in `neo4j.conf`. The calls use the default address, so the key should be sent and `metadata.authenticated` should be `True`.

Jev reports `confidence` as the margin between its two probabilities. The check allows a gap of 0.02, because Jev rounds its probabilities to two decimals.

In [12]:
if RUN_REAL_JEV:
    runs = [run_decide(STATE) for _ in range(REAL_JEV_RUNS)]
    results = [r for r, _ in runs]
    walls = [ms for _, ms in runs]

    check("real Jev: no errors", all(r["error_message"] is None for r in results),
          next((r["error_message"] for r in results if r["error_message"]), ""))
    check("real Jev: the key was sent", all(r["metadata"]["authenticated"] for r in results))
    check("real Jev: one attempt each", all(r["metadata"]["attempts"] == 1 for r in results))

    decision = decision_of(results[0])
    check("real Jev: the choice is one we offered", decision.get("choice") in ("Flag", "Pass"),
          decision.get("choice"))

    probs = list(decision.get("probabilities", {}).values())
    margin = abs(probs[0] - probs[1]) if len(probs) == 2 else None
    check("real Jev: confidence is the margin between the two probabilities",
          margin is not None and abs(decision.get("confidence", 0) - margin) <= 0.02,
          f"confidence {decision.get('confidence')}, margin {margin}")

    TIMINGS["real_jev_wall_ms"] = [round(w, 1) for w in walls]
    observe("real Jev: wall time per call (ms)",
            f"min {min(walls):.0f}, median {statistics.median(walls):.0f}, max {max(walls):.0f}")
else:
    skip("real Jev", "RUN_REAL_JEV is False")
    print("RUN_REAL_JEV is False -- no calls made to the hosted API.")

SKIPPED  real Jev -- RUN_REAL_JEV is False
RUN_REAL_JEV is False -- no calls made to the hosted API.


## 10. Summary

Every test with its status. A `FAIL` row names what went wrong in the detail column. `INFO` rows hold the measurements, and `SKIPPED` rows show what couldn't run and why.

In [13]:
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 200)

results_df = pd.DataFrame(RESULTS)
print(results_df["status"].value_counts().to_string())
print()
failed = results_df[results_df["status"] == "FAIL"]
print("No failures." if failed.empty else f"{len(failed)} failure(s):")
results_df

status
PASS       20
INFO        2
SKIPPED     1

No failures.


,test,status,detail
0,function jev.decide is registered,PASS,jev.decide
1,validation: choice criteria as an array,PASS,"validation: d: a choice question needs criteria as a map of label to description, not an array"
2,validation: no call was made,PASS,
3,validation: answers are null,PASS,
4,validation: unknown option name,PASS,"validation: unknown option(s): modle; known options are [connect_timeout_ms, max_retries, model, request_t..."
5,validation: blank state,PASS,validation: state must not be blank
6,validation: score criteria as a map,PASS,"validation: s: a score question needs criteria as an ordered array of level descriptions, not a map"
7,validation: retries out of range,PASS,validation: max_retries must be between 0 and 5
8,validation: missing instructions,PASS,validation: d: instructions are required
9,validation: url that is not http,PASS,validation: url must be an http or https URL with a host


## 11. Teardown

In [14]:
save_results()
driver.close()
print("Connection closed.")

Saved 23 results to results_decide.json
Connection closed.
